In [ ]:
import os, pandas as pd, torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

XLSX_PATH = "/content/Dataset_550_Nemotron.xlsx"
OUT_CSV   = "/content/nemotron_cot_output.csv"
PROG_CSV  = "/content/nemotron_cot_progress.csv"

MODEL_ID = "nvidia/Nemotron-4-Mini-Hindi-4B-Base"


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving Dataset_550_Nemotron.xlsx to Dataset_550_Nemotron.xlsx


In [ ]:
df = pd.read_excel(XLSX_PATH)
print("Rows:", len(df))
print(df.columns.tolist())
df.head(2)


Rows: 554
['PromptID', 'English', 'Hindi']


,PromptID,English,Hindi
0,PO1,So my therapist recommended to me that I shoul...,इसलिए मेरे चिकित्सक ने मुझे सलाह दी कि मुझे रे...
1,PO2,Hey all! I'd like to start this thread to mayb...,"अरे सभी अद्भुत लोग, मैं इस धागे को शुरू करना च..."


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(" Model loaded on:", model.device)


config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/18.1M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.55M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/8.38G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/324 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/8.38G [00:00<?, ?B/s]

✅ Model loaded on: cuda:0


In [ ]:
def make_cot_en(text: str) -> str:
    return f"""You are a supportive, safe mental-health assistant.

TASK:
Read the user message and reply with empathy and practical next steps.

IMPORTANT:
- Think step-by-step internally, but DO NOT show your reasoning.
- Do NOT mention that you are thinking or reasoning.
- Keep the response calm, non-judgmental, and actionable.
- If the message suggests self-harm risk, encourage reaching local emergency help and a trusted person.

USER MESSAGE:
{text}

OUTPUT:
Write only the final response (no analysis, no steps).
"""

def make_cot_hi(text: str) -> str:
    return f"""आप एक सहानुभूतिपूर्ण और सुरक्षित मानसिक-स्वास्थ्य सहायक हैं।

कार्य:
यूज़र के संदेश का जवाब सहानुभूति के साथ दें और व्यवहारिक अगले कदम बताएं।

महत्वपूर्ण:
- अंदर ही अंदर step-by-step सोचें, लेकिन reasoning/steps लिखना नहीं है।
- यह न लिखें कि आप सोच रहे हैं।
- जवाब शांत, non-judgmental और actionable हो।
- अगर self-harm risk लगे, तो आपातकालीन मदद/किसी भरोसेमंद व्यक्ति से तुरंत संपर्क करने को कहें।

यूज़र संदेश:
{text}

आउटपुट:
सिर्फ अंतिम जवाब लिखें (analysis/steps नहीं)।
"""


In [ ]:
@torch.inference_mode()
def generate(prompt: str, max_new_tokens=220, temperature=0.7, top_p=0.9) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    if text.startswith(prompt):
        text = text[len(prompt):].strip()
    return text.strip()


In [ ]:
done = set()
if os.path.exists(PROG_CSV):
    prog = pd.read_csv(PROG_CSV)
    done = set(prog["PromptID"].astype(str).tolist())
    print(" Resuming. Done:", len(done))
else:
    print(" Fresh run.")

rows_out = []
if os.path.exists(OUT_CSV):
    existing = pd.read_csv(OUT_CSV)
    rows_out = existing.to_dict("records")
    print(" Loaded existing output rows:", len(rows_out))

for _, row in tqdm(df.iterrows(), total=len(df)):
    pid = str(row["PromptID"])
    if pid in done:
        continue

    en_text = str(row["English"])
    hi_text = str(row["Hindi"])

    en_prompt = make_cot_en(en_text)
    hi_prompt = make_cot_hi(hi_text)

    en_resp = generate(en_prompt)
    hi_resp = generate(hi_prompt)

    rec = {
        "PromptID": pid,
        "English": en_text,
        "Hindi": hi_text,
        "EN_CoT_Prompt": en_prompt,
        "EN_CoT_Response": en_resp,
        "HI_CoT_Prompt": hi_prompt,
        "HI_CoT_Response": hi_resp,
    }
    rows_out.append(rec)
    done.add(pid)

    if len(done) % 10 == 0:
        pd.DataFrame(rows_out).to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
        pd.DataFrame({"PromptID": list(done)}).to_csv(PROG_CSV, index=False)
        print(f"💾 Saved checkpoint at {len(done)}")

pd.DataFrame(rows_out).to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
pd.DataFrame({"PromptID": list(done)}).to_csv(PROG_CSV, index=False)
print(" Completed. Output:", OUT_CSV)


✅ Fresh run.


  2%|▏         | 10/554 [02:03<1:43:15, 11.39s/it]

💾 Saved checkpoint at 10


  4%|▎         | 20/554 [04:12<1:56:22, 13.08s/it]

💾 Saved checkpoint at 20


...

💾 Saved checkpoint at 540


 99%|█████████▉| 550/554 [1:55:13<00:53, 13.26s/it]

💾 Saved checkpoint at 550


100%|██████████| 554/554 [1:56:06<00:00, 12.58s/it]

✅ Completed. Output: /content/nemotron_cot_output.csv


In [ ]:
import pandas as pd
from google.colab import files

df = pd.read_csv("/content/nemotron_cot_output.csv", encoding="utf-8-sig")
xlsx_path = "/content/nemotron_cot_output.xlsx"
df.to_excel(xlsx_path, index=False)
files.download(xlsx_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>